# Class Assignments & Submissions: Fabric Medallion Pipeline

**Bronze -> Silver -> Gold** on a Microsoft Fabric Lakehouse, with SQL answers to product
questions exposed as an API-style `{status, message, data}` contract.

**How to run in Fabric**
1. Create a Lakehouse (e.g. `lh_education`) and attach it to this notebook as the **default lakehouse**.
2. *Run all*. The first cells define the pipeline; the cells from **Run the pipeline** onwards execute it.

Runs unchanged on a laptop too (`PIPELINE_ENV=local`): tables become Parquet under `./lakehouse/Tables`.

> This notebook is generated from `src/pipeline/*.py` by `scripts/build_notebook.py`. Edit the source, not the notebook.

## 1. Configuration
Detects whether we are running inside a Microsoft Fabric notebook or locally,
and resolves every path from that. Settings can be overridden with environment
variables (see `.env.example`); nothing secret is hard-coded.

In [36]:
import os
import sys
from dataclasses import dataclass, field
from datetime import date

try:  # optional locally, never needed in Fabric
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:  # pragma: no cover
    pass


def running_in_fabric() -> bool:
    """Fabric notebooks ship `notebookutils` and mount the default lakehouse."""
    if os.getenv("PIPELINE_ENV", "").lower() == "local":
        return False
    return "notebookutils" in sys.modules or os.path.isdir("/lakehouse/default")


@dataclass
class PipelineConfig:
    """All runtime settings in one place."""

    is_fabric: bool = field(default_factory=running_in_fabric)
    # Fixed "today" so metrics are reproducible; override with AS_OF_DATE=YYYY-MM-DD
    as_of_date: str = field(default_factory=lambda: os.getenv("AS_OF_DATE", "2026-10-15"))
    seed: int = field(default_factory=lambda: int(os.getenv("DATA_SEED", "42")))
    local_root: str = field(default_factory=lambda: os.getenv("LAKEHOUSE_ROOT", "./lakehouse"))

    @property
    def files_fs(self) -> str:
        """Lakehouse 'Files' area as seen by plain Python (open, os, json)."""
        if self.is_fabric:
            return "/lakehouse/default/Files"
        return os.path.abspath(os.path.join(self.local_root, "Files"))

    @property
    def files_spark(self) -> str:
        """Lakehouse 'Files' area as seen by Spark readers."""
        return "Files" if self.is_fabric else self.files_fs

    @property
    def tables_dir(self) -> str:
        """Local stand-in for the lakehouse 'Tables' area (Spark warehouse dir)."""
        return os.path.abspath(os.path.join(self.local_root, "Tables"))

    @property
    def table_format(self) -> str:
        # Fabric lakehouse tables are Delta. Locally we use Parquet so the
        # project runs with plain `pip install pyspark` (no extra jars).
        return os.getenv("TABLE_FORMAT", "delta" if self.is_fabric else "parquet")

    @property
    def landing_fs(self) -> str:
        return os.path.join(self.files_fs, "landing")

    @property
    def landing_spark(self) -> str:
        return f"{self.files_spark}/landing"

    @property
    def exports_fs(self) -> str:
        return os.path.join(self.files_fs, "exports")

    @property
    def as_of(self) -> date:
        return date.fromisoformat(self.as_of_date)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 38, Finished, Available, Finished, False)

## 2. Spark session & Lakehouse helpers
In Fabric the `spark` session already exists and `saveAsTable` writes Delta
tables into the attached Lakehouse (`Tables/`). Locally we build a session whose
warehouse directory plays the role of `Tables/`.

In [37]:
import logging
import shutil
import sys

from pyspark.sql import DataFrame, SparkSession

log = logging.getLogger("pipeline")
if not log.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


def get_spark(cfg) -> SparkSession:
    if cfg.is_fabric:
        return SparkSession.builder.getOrCreate()
    return (
        SparkSession.builder.appName("fabric-edu-pipeline")
        .master("local[*]")
        .config("spark.sql.warehouse.dir", cfg.tables_dir)
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )


def reset_local_lakehouse(cfg) -> None:
    """Local only: wipe Tables/ so reruns are idempotent (Fabric overwrites in place)."""
    if not cfg.is_fabric:
        shutil.rmtree(cfg.tables_dir, ignore_errors=True)


def write_table(df: DataFrame, name: str, cfg, mode: str = "overwrite") -> DataFrame:
    """Persist a DataFrame as a managed Lakehouse table (Delta in Fabric)."""
    (
        df.write.format(cfg.table_format)
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(name)
    )
    count = df.sparkSession.table(name).count()
    log.info("wrote %-38s %6d rows", name, count)
    return df.sparkSession.table(name)


def show(df: DataFrame, n: int = 20) -> None:
    """Render as a standard HTML table, so outputs survive .ipynb download and GitHub."""
    pdf = df.limit(n).toPandas()
    try:
        from IPython.display import HTML, display as ipy_display
        ipy_display(HTML(pdf.to_html(index=False)))
    except ImportError:
        print(pdf.to_string(index=False))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 39, Finished, Available, Finished, False)

## 3. Simulated source systems
Generates four sources the way they would arrive from real systems:

| Source | Format | Simulates |
|---|---|---|
| `courses.csv` | CSV | Course catalogue export |
| `assignments.csv` | CSV | LMS assignment export |
| `students.json` | JSON Lines, nested `enrolled_courses` array | Student Information System API |
| `submissions.json` | JSON Lines event stream | LMS submission events |

The data is deterministic (fixed seed) and **intentionally dirty** so the
Silver layer has real work to do: duplicate rows, resubmissions, mixed date
formats, lower-case / padded IDs, orphan keys, negative scores, unparseable
timestamps, a student submitting to a course they are not enrolled in, etc.

In [38]:
import csv
import json
import os
import random
from datetime import datetime, timedelta

COURSES = [
    ("C001", "Data Engineering Fundamentals", "Computer Science", "Dr. Ayesha Khan"),
    ("C002", "Machine Learning I", "Computer Science", "Dr. Layla Haddad"),
    ("C003", "Statistics for Analytics", "Mathematics", "Prof. Marco Rossi"),
    ("C004", "Cloud Architecture on Azure", "Information Systems", "Dr. Priya Nair"),
    ("C005", "Business Intelligence & Visualisation", "Business", "Prof. Daniel Okafor"),
]
ASSIGNMENT_TITLES = ["Lab", "Project Milestone", "Final Project"]
FIRST = ["Aarav", "Maya", "Omar", "Sofia", "Liam", "Zara", "Noah", "Ananya", "Lucas", "Fatima",
         "Ethan", "Isla", "Arjun", "Hana", "Leo", "Chloe", "Yusuf", "Mia", "Kabir", "Elena"]
LAST = ["Sharma", "Haddad", "Rossi", "Chen", "Okafor", "Nair", "Garcia", "Kim", "Smith", "Ali",
        "Mehta", "Novak", "Silva", "Brown", "Iyer", "Costa"]
TERM_START = datetime(2026, 9, 1)


def _iso(ts: datetime) -> str:
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def build_source_data(seed: int = 42) -> dict:
    """Return the four raw datasets as lists of dicts (before any cleaning)."""
    rnd = random.Random(seed)

    # ---------- courses ----------
    courses = [
        {"course_id": cid, "course_name": name, "department": dept,
         "instructor": instr, "term": "Fall 2026", "credits": rnd.choice([3, 4])}
        for cid, name, dept, instr in COURSES
    ]
    courses[2]["course_name"] = "  Statistics for Analytics  "   # padded text
    courses.append(dict(courses[3]))                                # exact duplicate row

    # ---------- assignments: 3 per course (2 already due, 1 in the future) ----------
    assignments = []
    for k, (cid, *_rest) in enumerate(COURSES):
        dues = [datetime(2026, 9, 20) + timedelta(days=3 * k),
                datetime(2026, 10, 4) + timedelta(days=3 * k),
                datetime(2026, 10, 25) + timedelta(days=3 * k)]
        for j, due in enumerate(dues, start=1):
            due = due.replace(hour=23, minute=59)
            assignments.append({
                "assignment_id": f"Assignment_{k + 1}0{j}",
                "course_id": cid,
                "title": f"{ASSIGNMENT_TITLES[j - 1]} {j}",
                "due_date": due.strftime("%Y-%m-%d %H:%M:%S"),
                "max_score": "100",
            })
    assignments[4]["due_date"] = "2026/10/07 23:59"           # different date format
    assignments.append({"assignment_id": "Assignment_901", "course_id": "C999",  # orphan course
                        "title": "Legacy Quiz", "due_date": "2026-09-30 23:59:00",
                        "max_score": "50"})

    # ---------- students (nested enrolments) ----------
    students = []
    for i in range(1, 61):
        first, last = rnd.choice(FIRST), rnd.choice(LAST)
        enrolled = sorted(rnd.sample([c[0] for c in COURSES], rnd.choice([2, 3])))
        if i <= 20 and "C001" not in enrolled:  # make sure C001 has a healthy roster
            enrolled = sorted(set(enrolled[:-1] + ["C001"]))
        students.append({
            "student_id": f"S{i:04d}",
            "first_name": first,
            "last_name": last,
            "email": f"{first}.{last}{i}@university.edu".lower(),
            "phone": f"+1-555-{rnd.randint(100, 999)}-{rnd.randint(1000, 9999)}",
            "date_of_birth": (datetime(2000, 1, 1) + timedelta(days=rnd.randint(0, 2500))).strftime("%Y-%m-%d"),
            "enrollment_date": (TERM_START - timedelta(days=rnd.randint(0, 20))).strftime("%Y-%m-%d"),
            "enrolled_courses": enrolled,
            "updated_at": _iso(TERM_START - timedelta(days=30)),
        })
    students[11]["student_id"] = " s0012 "                     # padded + lower-case key
    students[4]["email"] = students[4]["email"].upper()         # inconsistent casing
    students[8]["phone"] = None                                 # missing value
    students[15]["enrolled_courses"] = [c.lower() for c in students[15]["enrolled_courses"]]
    dup = dict(students[6])                                     # later CDC update of S0007
    dup["email"] = dup["email"].replace("@university.edu", "@alumni.university.edu")
    dup["updated_at"] = _iso(TERM_START + timedelta(days=10))
    students.append(dup)

    # ---------- submissions ----------
    as_of = datetime(2026, 10, 15)
    roster = {}
    for s in students[:60]:
        for c in s["enrolled_courses"]:
            roster.setdefault(c.upper(), []).append(s["student_id"].strip().upper())
    # The first four C001 students never submit Assignment_101 (guaranteed answer for Q1)
    never_submit_101 = set(sorted(roster["C001"])[:4])

    submissions, n = [], 0

    def add(student_id, assignment_id, ts, score, fmt_iso=True):
        nonlocal n
        n += 1
        submissions.append({
            "submission_id": f"SUB{n:05d}",
            "student_id": student_id,
            "assignment_id": assignment_id,
            "submitted_at": _iso(ts) if fmt_iso else ts.strftime("%Y-%m-%d %H:%M:%S"),
            "score": score,
            "channel": rnd.choice(["web", "mobile", "lms_api"]),
        })

    for a in assignments[:15]:
        due = datetime.strptime(a["due_date"].replace("/", "-")[:16], "%Y-%m-%d %H:%M")
        is_past_due = due < as_of
        for sid in roster[a["course_id"]]:
            if a["assignment_id"] == "Assignment_101" and sid in never_submit_101:
                continue
            p_submit = 0.82 if is_past_due else 0.30
            if rnd.random() > p_submit:
                continue
            late = is_past_due and rnd.random() < 0.18
            ts = due + timedelta(hours=rnd.randint(2, 96)) if late else due - timedelta(hours=rnd.randint(1, 120))
            if not is_past_due:
                ts = min(ts, as_of - timedelta(hours=rnd.randint(1, 48)))
            score = rnd.randint(55, 100)
            add(sid, a["assignment_id"], ts, score, fmt_iso=rnd.random() > 0.15)
            if rnd.random() < 0.10:  # resubmission: later attempt with a new score
                add(sid, a["assignment_id"], ts + timedelta(hours=rnd.randint(1, 30)), min(100, score + 7))

    # ---- inject data-quality problems ----
    submissions.append(dict(submissions[3]))                     # exact duplicate event
    submissions[10]["student_id"] = " " + submissions[10]["student_id"].lower()  # dirty key
    add("S9999", "Assignment_102", datetime(2026, 10, 1, 10), 88)  # orphan student
    add("S0002", "Assignment_999", datetime(2026, 10, 1, 10), 70)  # orphan assignment
    submissions[20]["submitted_at"] = "not-a-date"               # unparseable timestamp
    submissions[30]["score"] = -5                                # out-of-range score
    outsider = next(s["student_id"] for s in students[:60] if "C005" not in s["enrolled_courses"])
    add(outsider.strip().upper(), "Assignment_501", datetime(2026, 9, 30, 9), 90)  # not enrolled
    rnd.shuffle(submissions)                                     # events arrive out of order

    return {"courses": courses, "assignments": assignments,
            "students": students, "submissions": submissions}


def write_source_files(target_dir: str, seed: int = 42) -> dict:
    """Write the raw files exactly as a source system would drop them."""
    os.makedirs(target_dir, exist_ok=True)
    data = build_source_data(seed)
    paths = {}
    for name in ("courses", "assignments"):
        path = os.path.join(target_dir, f"{name}.csv")
        with open(path, "w", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(data[name][0].keys()))
            writer.writeheader()
            writer.writerows(data[name])
        paths[name] = path
    for name in ("students", "submissions"):
        path = os.path.join(target_dir, f"{name}.json")
        with open(path, "w", encoding="utf-8") as fh:
            for row in data[name]:
                fh.write(json.dumps(row) + "\n")
        paths[name] = path
    return {k: (v, len(data[k])) for k, v in paths.items()}

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 40, Finished, Available, Finished, False)

## 4. Bronze layer: raw ingestion
* The raw files stay untouched in `Files/landing/` (the immutable copy).
* Each file is loaded **as-is, every column as string** into a `bronze_*`
  Delta table, plus lineage columns: `_source_file`, `_ingested_at`, `_batch_id`.
* No cleaning happens here: Bronze must be replayable if Silver logic changes.

In [39]:
import uuid

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType, StructField, StructType


BRONZE_SOURCES = {
    # table name        file                 reader
    "bronze_courses": ("courses.csv", "csv"),
    "bronze_assignments": ("assignments.csv", "csv"),
    "bronze_students": ("students.json", "json"),
    "bronze_submissions": ("submissions.json", "json"),
}

# Explicit raw schemas for JSON: everything string, so a bad value never breaks ingestion
RAW_JSON_SCHEMAS = {
    "students.json": StructType(
        [StructField(c, StringType()) for c in
         ["student_id", "first_name", "last_name", "email", "phone",
          "date_of_birth", "enrollment_date"]]
        + [StructField("enrolled_courses", ArrayType(StringType())),
           StructField("updated_at", StringType())]
    ),
    "submissions.json": StructType(
        [StructField(c, StringType()) for c in
         ["submission_id", "student_id", "assignment_id", "submitted_at", "score", "channel"]]
    ),
}


def read_raw(spark, path: str, fmt: str):
    if fmt == "csv":
        return spark.read.option("header", True).option("inferSchema", False).csv(path)
    schema = RAW_JSON_SCHEMAS[path.rsplit("/", 1)[-1]]
    return spark.read.schema(schema).option("mode", "PERMISSIVE").json(path)


def run_bronze(spark, cfg) -> dict:
    batch_id = uuid.uuid4().hex[:12]
    counts = {}
    for table, (file_name, fmt) in BRONZE_SOURCES.items():
        df = (
            read_raw(spark, f"{cfg.landing_spark}/{file_name}", fmt)
            .withColumn("_source_file", F.lit(file_name))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_batch_id", F.lit(batch_id))
        )
        counts[table] = write_table(df, table, cfg).count()
    return counts

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 41, Finished, Available, Finished, False)

## 5. Silver layer: cleaned, typed, conformed, joined
| Table | Grain | Key rules |
|---|---|---|
| `silver_courses` | course | trim text, upper-case IDs, drop duplicate rows |
| `silver_students` | student | normalise IDs/emails, CDC dedup (latest `updated_at` wins) |
| `silver_enrollments` | student × course | exploded from the nested JSON array |
| `silver_assignments` | assignment | multi-format date parsing, FK to course enforced |
| `silver_submissions` | student × assignment | resubmissions collapsed: latest score, first submit time |
| `silver_student_assignment_status` | student × assignment | **the student↔submission mapping**: every assignment a student owes, with its status |
| `silver_quarantine` | rejected record | invalid rows are kept with a reason instead of silently dropped |

Business rules (documented so the Gold numbers are explainable):
* A resubmission keeps the **latest score**; lateness is judged on the **first** attempt.
* An assignment is *due* when `due_at < as_of_date`.
* Status: `SUBMITTED_ON_TIME`, `SUBMITTED_LATE`, `MISSING_OVERDUE` (due, nothing submitted), `PENDING` (not due yet).


In [40]:
from pyspark.sql import Window
from pyspark.sql import functions as F


TS_FORMATS = ["yyyy-MM-dd'T'HH:mm:ss'Z'", "yyyy-MM-dd HH:mm:ss", "yyyy/MM/dd HH:mm", "yyyy-MM-dd HH:mm"]


def clean_key(col_name: str):
    """' s0012 ' -> 'S0012'"""
    return F.upper(F.trim(F.col(col_name)))


def parse_ts(col_name: str):
    """Try every known source format; unparseable values become NULL (never fail the job)."""
    return F.coalesce(*[F.try_to_timestamp(F.col(col_name), F.lit(fmt)) for fmt in TS_FORMATS])


def quarantine(df, entity: str, key_col: str, reason_col: str = "_reject_reason"):
    """Shape rejected rows into a common quarantine schema."""
    payload_cols = [c for c in df.columns if not c.startswith("_") and c != reason_col]
    return df.select(
        F.lit(entity).alias("entity"),
        F.col(key_col).cast("string").alias("record_key"),
        F.col(reason_col).alias("reject_reason"),
        F.to_json(F.struct(*payload_cols)).alias("raw_record"),
        F.current_timestamp().alias("quarantined_at"),
    )


def build_courses(bronze):
    return (
        bronze.select(
            clean_key("course_id").alias("course_id"),
            F.trim("course_name").alias("course_name"),
            F.trim("department").alias("department"),
            F.trim("instructor").alias("instructor"),
            F.trim("term").alias("term"),
            F.col("credits").cast("int").alias("credits"),
        )
        .filter(F.col("course_id").isNotNull())
        .dropDuplicates(["course_id"])
    )


def build_students(bronze):
    latest_first = Window.partitionBy("student_id").orderBy(F.col("updated_at").desc_nulls_last())
    return (
        bronze.select(
            clean_key("student_id").alias("student_id"),
            F.initcap(F.trim("first_name")).alias("first_name"),
            F.initcap(F.trim("last_name")).alias("last_name"),
            F.lower(F.trim("email")).alias("email"),
            F.trim("phone").alias("phone"),
            F.to_date("date_of_birth").alias("date_of_birth"),
            F.to_date("enrollment_date").alias("enrollment_date"),
            F.transform("enrolled_courses", lambda c: F.upper(F.trim(c))).alias("enrolled_courses"),
            parse_ts("updated_at").alias("updated_at"),
        )
        .filter(F.col("student_id").isNotNull())
        .withColumn("_rn", F.row_number().over(latest_first))
        .filter("_rn = 1")
        .drop("_rn")
    )


def build_enrollments(students, courses):
    return (
        students.select("student_id", F.explode("enrolled_courses").alias("course_id"))
        .join(courses.select("course_id"), "course_id", "left_semi")
        .dropDuplicates(["student_id", "course_id"])
        .select("student_id", "course_id")
    )


def build_assignments(bronze, courses):
    typed = bronze.select(
        F.trim("assignment_id").alias("assignment_id"),
        clean_key("course_id").alias("course_id"),
        F.trim("title").alias("title"),
        parse_ts("due_date").alias("due_at"),
        F.col("max_score").cast("int").alias("max_score"),
    ).join(courses.select("course_id", F.lit(True).alias("_course_ok")), "course_id", "left")
    flagged = typed.withColumn(
        "_reject_reason",
        F.when(F.col("_course_ok").isNull(), "UNKNOWN_COURSE")
        .when(F.col("due_at").isNull(), "INVALID_DUE_DATE"),
    )
    good = flagged.filter(F.col("_reject_reason").isNull()).drop("_course_ok", "_reject_reason")
    bad = quarantine(flagged.filter(F.col("_reject_reason").isNotNull()), "assignments", "assignment_id")
    return good.dropDuplicates(["assignment_id"]), bad


def build_submissions(bronze, students, assignments, enrollments):
    events = bronze.select(
        F.upper(F.trim("submission_id")).alias("submission_id"),
        clean_key("student_id").alias("student_id"),
        F.trim("assignment_id").alias("assignment_id"),
        parse_ts("submitted_at").alias("submitted_at"),
        F.col("score").cast("double").alias("raw_score"),
        F.lower(F.trim("channel")).alias("channel"),
    ).dropDuplicates(["submission_id"])  # identical replayed events

    checked = (
        events.join(students.select("student_id", F.lit(True).alias("_student_ok")), "student_id", "left")
        .join(assignments.select("assignment_id", "course_id", "max_score"), "assignment_id", "left")
        .join(enrollments.withColumn("_enrolled", F.lit(True)), ["student_id", "course_id"], "left")
        .withColumn(
            "_reject_reason",
            F.when(F.col("submitted_at").isNull(), "INVALID_TIMESTAMP")
            .when(F.col("_student_ok").isNull(), "UNKNOWN_STUDENT")
            .when(F.col("course_id").isNull(), "UNKNOWN_ASSIGNMENT")
            .when(F.col("_enrolled").isNull(), "NOT_ENROLLED"),
        )
    )
    rejected = quarantine(checked.filter(F.col("_reject_reason").isNotNull()), "submissions", "submission_id")

    valid = checked.filter(F.col("_reject_reason").isNull()).withColumn(
        "score",
        F.when(F.col("raw_score").between(0, F.col("max_score")), F.col("raw_score")),
    ).withColumn("score_valid", F.col("score").isNotNull())

    by_attempt = Window.partitionBy("student_id", "assignment_id").orderBy(F.col("submitted_at").desc())
    agg = Window.partitionBy("student_id", "assignment_id")
    collapsed = (
        valid.withColumn("first_submitted_at", F.min("submitted_at").over(agg))
        .withColumn("attempt_count", F.count("*").over(agg))
        .withColumn("_rn", F.row_number().over(by_attempt))
        .filter("_rn = 1")
        .select(
            "submission_id", "student_id", "assignment_id", "course_id",
            "first_submitted_at", F.col("submitted_at").alias("last_submitted_at"),
            "attempt_count", "score", "score_valid", "channel",
        )
    )
    return collapsed, rejected


def build_student_assignment_status(enrollments, assignments, submissions, as_of_date: str):
    as_of_ts = F.to_timestamp(F.lit(as_of_date))
    owed = enrollments.join(assignments.select("assignment_id", "course_id", "due_at"), "course_id")
    j = owed.join(
        submissions.drop("course_id"), ["student_id", "assignment_id"], "left"
    )
    is_submitted = F.col("submission_id").isNotNull()
    is_late = is_submitted & (F.col("first_submitted_at") > F.col("due_at"))
    status = (
        F.when(is_submitted & ~is_late, "SUBMITTED_ON_TIME")
        .when(is_late, "SUBMITTED_LATE")
        .when(F.col("due_at") < as_of_ts, "MISSING_OVERDUE")
        .otherwise("PENDING")
    )
    return j.select(
        "student_id", "course_id", "assignment_id", "due_at", "submission_id",
        "first_submitted_at", "last_submitted_at", "attempt_count", "score",
        (F.col("due_at") < as_of_ts).alias("is_due"),
        is_submitted.alias("is_submitted"),
        F.coalesce(is_late, F.lit(False)).alias("is_late"),
        status.alias("status"),
        F.when(status == "MISSING_OVERDUE", F.datediff(as_of_ts, F.col("due_at")))
        .when(status == "SUBMITTED_LATE", F.datediff(F.col("first_submitted_at"), F.col("due_at")))
        .otherwise(0).alias("days_overdue"),
        F.to_date(F.lit(as_of_date)).alias("as_of_date"),
    )


def run_silver(spark, cfg) -> dict:
    courses = write_table(build_courses(spark.table("bronze_courses")), "silver_courses", cfg)
    students_full = build_students(spark.table("bronze_students"))
    enrollments = write_table(build_enrollments(students_full, courses), "silver_enrollments", cfg)
    students = write_table(students_full.drop("enrolled_courses"), "silver_students", cfg)

    assignments_df, bad_assignments = build_assignments(spark.table("bronze_assignments"), courses)
    assignments = write_table(assignments_df, "silver_assignments", cfg)

    submissions_df, bad_submissions = build_submissions(
        spark.table("bronze_submissions"), students, assignments, enrollments
    )
    submissions = write_table(submissions_df, "silver_submissions", cfg)
    write_table(bad_assignments.unionByName(bad_submissions), "silver_quarantine", cfg)

    status = build_student_assignment_status(enrollments, assignments, submissions, cfg.as_of_date)
    write_table(status, "silver_student_assignment_status", cfg)
    return {t: spark.table(t).count() for t in [
        "silver_courses", "silver_students", "silver_enrollments", "silver_assignments",
        "silver_submissions", "silver_quarantine", "silver_student_assignment_status"]}

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 42, Finished, Available, Finished, False)

## 6. SQL loader
Gold tables and the Part 2 questions live in `sql/*.sql` files so they can be
reviewed, diffed and reused on the Fabric SQL analytics endpoint. In the
notebook build, the files are embedded below so the notebook is self-contained.

In [41]:
from pathlib import Path
from string import Template

EMBEDDED_SQL: dict = {}  # populated by scripts/build_notebook.py


def _sql_dir():
    try:
        return Path(__file__).resolve().parents[2] / "sql"
    except NameError:  # inside a notebook there is no __file__
        return Path("sql")


def load_sql(name: str, **params) -> str:
    text = EMBEDDED_SQL.get(name) or (_sql_dir() / name).read_text(encoding="utf-8")
    return Template(text).safe_substitute(**params)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 43, Finished, Available, Finished, False)

In [42]:
# SQL files from the repo's sql/ folder, embedded so this notebook is self-contained

EMBEDDED_SQL["gold/gold_assignment_summary.sql"] = """
-- Gold: one row per assignment, handy for dashboards and "who hasn't submitted X" lookups.
SELECT
    a.assignment_id,
    a.course_id,
    a.title,
    a.due_at,
    COUNT(sa.student_id)                                                    AS students_assigned,
    SUM(CASE WHEN sa.is_submitted THEN 1 ELSE 0 END)                        AS submitted,
    SUM(CASE WHEN NOT sa.is_submitted THEN 1 ELSE 0 END)                    AS not_submitted,
    SUM(CASE WHEN sa.status = 'SUBMITTED_LATE' THEN 1 ELSE 0 END)           AS late,
    ROUND(100.0 * SUM(CASE WHEN sa.is_submitted THEN 1 ELSE 0 END) / NULLIF(COUNT(sa.student_id), 0), 2) AS submission_rate_pct,
    ROUND(AVG(sa.score), 2)                                                 AS avg_score,
    MAX(sa.is_due)                                                          AS is_due,
    current_timestamp()                                                     AS _refreshed_at
FROM silver_assignments a
LEFT JOIN silver_student_assignment_status sa
       ON sa.assignment_id = a.assignment_id
GROUP BY a.assignment_id, a.course_id, a.title, a.due_at
"""

EMBEDDED_SQL["gold/gold_course_submission_rate.sql"] = """
-- Gold: submission rate per course.
-- expected  = enrolled students x assignments already due (as of the run date)
-- submitted = of those, how many have a valid submission (on time or late)
WITH course_facts AS (
    SELECT
        c.course_id,
        c.course_name,
        c.department,
        c.instructor,
        COUNT(DISTINCT sa.student_id)                                         AS enrolled_students,
        COUNT(DISTINCT CASE WHEN sa.is_due THEN sa.assignment_id END)         AS assignments_due,
        SUM(CASE WHEN sa.is_due THEN 1 ELSE 0 END)                            AS expected_submissions,
        SUM(CASE WHEN sa.is_due AND sa.is_submitted THEN 1 ELSE 0 END)        AS actual_submissions,
        SUM(CASE WHEN sa.status = 'SUBMITTED_ON_TIME' AND sa.is_due THEN 1 ELSE 0 END) AS on_time_submissions,
        SUM(CASE WHEN sa.status = 'SUBMITTED_LATE' THEN 1 ELSE 0 END)         AS late_submissions,
        SUM(CASE WHEN sa.status = 'MISSING_OVERDUE' THEN 1 ELSE 0 END)        AS missing_submissions,
        ROUND(AVG(sa.score), 2)                                               AS avg_score
    FROM silver_courses c
    LEFT JOIN silver_student_assignment_status sa
           ON sa.course_id = c.course_id
    GROUP BY c.course_id, c.course_name, c.department, c.instructor
)
SELECT
    *,
    ROUND(100.0 * actual_submissions  / NULLIF(expected_submissions, 0), 2) AS submission_rate_pct,
    ROUND(100.0 * on_time_submissions / NULLIF(expected_submissions, 0), 2) AS on_time_rate_pct,
    current_timestamp()                                                     AS _refreshed_at
FROM course_facts
"""

EMBEDDED_SQL["gold/gold_student_overdue_metrics.sql"] = """
-- Gold: overdue metrics per student (one row per student, zero-filled).
-- overdue_assignment_count = assignments past their due date with NO submission.
-- late_submission_count    = assignments that were submitted, but after the due date.
SELECT
    s.student_id,
    s.first_name,
    s.last_name,
    COUNT(sa.assignment_id)                                                 AS assignments_total,
    SUM(CASE WHEN sa.is_due THEN 1 ELSE 0 END)                              AS assignments_due,
    SUM(CASE WHEN sa.is_submitted THEN 1 ELSE 0 END)                        AS assignments_submitted,
    SUM(CASE WHEN sa.status = 'MISSING_OVERDUE' THEN 1 ELSE 0 END)          AS overdue_assignment_count,
    SUM(CASE WHEN sa.status = 'SUBMITTED_LATE'  THEN 1 ELSE 0 END)          AS late_submission_count,
    COALESCE(MAX(CASE WHEN sa.status = 'MISSING_OVERDUE' THEN sa.days_overdue END), 0) AS max_days_overdue,
    CASE
        WHEN SUM(CASE WHEN sa.status = 'MISSING_OVERDUE' THEN 1 ELSE 0 END) >= 2 THEN 'HIGH'
        WHEN SUM(CASE WHEN sa.status IN ('MISSING_OVERDUE', 'SUBMITTED_LATE') THEN 1 ELSE 0 END) >= 1 THEN 'MEDIUM'
        ELSE 'LOW'
    END                                                                     AS risk_band,
    current_timestamp()                                                     AS _refreshed_at
FROM silver_students s
LEFT JOIN silver_student_assignment_status sa
       ON sa.student_id = s.student_id
GROUP BY s.student_id, s.first_name, s.last_name
"""

EMBEDDED_SQL["queries/q1_students_not_submitted_assignment_101.sql"] = """
-- Q1: Students who have NOT submitted "Assignment_101".
-- Only students enrolled in the course that owns the assignment are considered
-- (a student outside the course was never expected to submit it).
SELECT
    st.student_id,
    st.first_name,
    st.last_name,
    e.course_id,
    a.assignment_id,
    a.due_at,
    DATEDIFF(DATE'${as_of_date}', a.due_at) AS days_past_due
FROM silver_enrollments e
JOIN silver_assignments a  ON a.course_id  = e.course_id
JOIN silver_students    st ON st.student_id = e.student_id
WHERE a.assignment_id = '${assignment_id}'
  AND NOT EXISTS (
        SELECT 1
        FROM silver_submissions sub
        WHERE sub.student_id    = e.student_id
          AND sub.assignment_id = a.assignment_id
  )
ORDER BY st.student_id
"""

EMBEDDED_SQL["queries/q2_submission_rate_per_course.sql"] = """
-- Q2: Submission rate per course (%), from the Gold table.
SELECT
    course_id,
    course_name,
    enrolled_students,
    assignments_due,
    expected_submissions,
    actual_submissions,
    submission_rate_pct,
    on_time_rate_pct
FROM gold_course_submission_rate
ORDER BY submission_rate_pct DESC, course_id
"""

EMBEDDED_SQL["queries/q3_overdue_count_per_student.sql"] = """
-- Q3: Overdue assignment count per student (past due, nothing submitted).
SELECT
    student_id,
    first_name,
    last_name,
    assignments_due,
    overdue_assignment_count,
    late_submission_count,
    max_days_overdue,
    risk_band
FROM gold_student_overdue_metrics
ORDER BY overdue_assignment_count DESC, late_submission_count DESC, student_id
"""

print(f'{len(EMBEDDED_SQL)} SQL definitions loaded')

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 44, Finished, Available, Finished, False)

6 SQL definitions loaded


## 7. Gold layer: analytical tables
Built with Spark SQL from the Silver model. These are the tables BI, the SQL
analytics endpoint, and downstream AI/ML features read from.

In [43]:
GOLD_TABLES = {
    "gold_course_submission_rate": "gold/gold_course_submission_rate.sql",
    "gold_student_overdue_metrics": "gold/gold_student_overdue_metrics.sql",
    "gold_assignment_summary": "gold/gold_assignment_summary.sql",
}


def run_gold(spark, cfg) -> dict:
    return {
        table: write_table(spark.sql(load_sql(sql_file)), table, cfg).count()
        for table, sql_file in GOLD_TABLES.items()
    }

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 45, Finished, Available, Finished, False)

## 8. Part 2: product questions as an API-style contract
Every answer is returned in the same envelope so a web/API layer (or an AI
agent tool call) can consume it without special cases:
```json
{"status": "success" | "error", "message": "<human readable>", "data": [ ...rows ]}
```

In [44]:
import json
import os
from datetime import date, datetime
from decimal import Decimal


QUERIES = {
    "students_not_submitted_assignment_101": (
        "queries/q1_students_not_submitted_assignment_101.sql",
        "{count} student(s) have not submitted {assignment_id}",
    ),
    "submission_rate_per_course": (
        "queries/q2_submission_rate_per_course.sql",
        "Submission rate for {count} course(s) as of {as_of_date}",
    ),
    "overdue_count_per_student": (
        "queries/q3_overdue_count_per_student.sql",
        "Overdue assignment counts for {count} student(s) as of {as_of_date}",
    ),
}


def _jsonable(value):
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return float(value)
    return value


def to_records(df) -> list:
    return [{k: _jsonable(v) for k, v in row.asDict(recursive=True).items()} for row in df.collect()]


def api_response(status: str, message: str, data=None) -> dict:
    return {"status": status, "message": message, "data": data if data is not None else []}


def run_query(spark, cfg, name: str, **params) -> dict:
    """Execute a named product question and wrap it in the {status, message, data} envelope."""
    params = {"as_of_date": cfg.as_of_date, "assignment_id": "Assignment_101", **params}
    try:
        if name not in QUERIES:
            return api_response("error", f"Unknown query '{name}'. Available: {sorted(QUERIES)}")
        sql_file, message = QUERIES[name]
        rows = to_records(spark.sql(load_sql(sql_file, **params)))
        return api_response("success", message.format(count=len(rows), **params), rows)
    except Exception as exc:  # the envelope must hold even on failure
        return api_response("error", f"{type(exc).__name__}: {exc}")


def export_api_responses(spark, cfg) -> dict:
    out_dir = os.path.join(cfg.exports_fs, "api")
    os.makedirs(out_dir, exist_ok=True)
    results = {}
    for name in QUERIES:
        payload = run_query(spark, cfg, name)
        with open(os.path.join(out_dir, f"{name}.json"), "w", encoding="utf-8") as fh:
            json.dump(payload, fh, indent=2)
        results[name] = payload
    return results


def export_gold_csv(spark, cfg, tables) -> list:
    out_dir = os.path.join(cfg.exports_fs, "gold")
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for table in tables:
        path = os.path.join(out_dir, f"{table}.csv")
        spark.table(table).toPandas().to_csv(path, index=False)  # Gold tables are small aggregates
        paths.append(path)
    return paths

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 46, Finished, Available, Finished, False)

## 9. Data-quality gates
Each check is a SQL query that must return **0** offending rows (or `True`).
The orchestrator fails the run if any check fails, so a bad batch never
reaches consumers. The same checks run in CI.

In [45]:
DQ_CHECKS = {
    # ---- uniqueness (primary keys) ----
    "pk_silver_students": "SELECT student_id FROM silver_students GROUP BY student_id HAVING COUNT(*) > 1",
    "pk_silver_courses": "SELECT course_id FROM silver_courses GROUP BY course_id HAVING COUNT(*) > 1",
    "pk_silver_assignments": "SELECT assignment_id FROM silver_assignments GROUP BY assignment_id HAVING COUNT(*) > 1",
    "pk_silver_submissions": """SELECT student_id, assignment_id FROM silver_submissions
                                GROUP BY student_id, assignment_id HAVING COUNT(*) > 1""",
    "pk_status": """SELECT student_id, assignment_id FROM silver_student_assignment_status
                    GROUP BY student_id, assignment_id HAVING COUNT(*) > 1""",
    # ---- not null keys ----
    "nn_submission_keys": """SELECT * FROM silver_submissions
                             WHERE student_id IS NULL OR assignment_id IS NULL OR first_submitted_at IS NULL""",
    # ---- referential integrity ----
    "fk_enrollment_student": """SELECT e.* FROM silver_enrollments e
                                LEFT ANTI JOIN silver_students s ON s.student_id = e.student_id""",
    "fk_submission_assignment": """SELECT x.* FROM silver_submissions x
                                   LEFT ANTI JOIN silver_assignments a ON a.assignment_id = x.assignment_id""",
    # ---- business-rule sanity ----
    "rate_between_0_100": """SELECT * FROM gold_course_submission_rate
                             WHERE submission_rate_pct < 0 OR submission_rate_pct > 100""",
    "score_in_range": """SELECT x.* FROM silver_submissions x JOIN silver_assignments a USING (assignment_id)
                         WHERE x.score < 0 OR x.score > a.max_score""",
    "overdue_not_negative": "SELECT * FROM gold_student_overdue_metrics WHERE overdue_assignment_count < 0",
    # ---- reconciliation between layers ----
    "every_student_in_gold": """SELECT s.student_id FROM silver_students s
                                LEFT ANTI JOIN gold_student_overdue_metrics g ON g.student_id = s.student_id""",
    "submissions_reconcile": """SELECT 1 FROM (SELECT COUNT(*) n FROM silver_submissions) a,
                                (SELECT SUM(CASE WHEN is_submitted THEN 1 ELSE 0 END) n
                                 FROM silver_student_assignment_status) b WHERE a.n <> b.n""",
}

NON_EMPTY_TABLES = [
    "bronze_students", "bronze_courses", "bronze_assignments", "bronze_submissions",
    "silver_students", "silver_courses", "silver_assignments", "silver_submissions",
    "silver_enrollments", "silver_student_assignment_status",
    "gold_course_submission_rate", "gold_student_overdue_metrics", "gold_assignment_summary",
]


class DataQualityError(RuntimeError):
    pass


def run_quality_checks(spark, raise_on_failure: bool = True) -> list:
    results = []
    for table in NON_EMPTY_TABLES:
        n = spark.table(table).count()
        results.append({"check": f"non_empty_{table}", "passed": n > 0, "failing_rows": 0 if n else 1})
    for name, sql in DQ_CHECKS.items():
        bad = spark.sql(sql).count()
        results.append({"check": name, "passed": bad == 0, "failing_rows": bad})
    failed = [r["check"] for r in results if not r["passed"]]
    if failed and raise_on_failure:
        raise DataQualityError(f"Data-quality checks failed: {failed}")
    return results

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 47, Finished, Available, Finished, False)

## 10. Bonus: security & governance (mocked in Spark, real T-SQL in `sql/security/`)
* **Data masking**: email, phone, date of birth and surname are masked for any
  role without PII clearance. A masked `gold_dim_student` is published for broad use,
  with a hashed `student_key` so ML features can join without seeing identity.
* **Row-level security**: an instructor only sees rows for the courses they teach.
* **Column-level security**: the ML service account never receives identity columns at all.
* **Metadata tagging / catalog**: every table gets layer, owner, sensitivity and PII
  tags as table properties, and a `data_catalog.json` is exported.
  (In Fabric this maps to Purview sensitivity labels + endorsement.)

In [46]:
import json
import os

from pyspark.sql import functions as F


# ---------- masking rules ----------
MASKS = {
    "email": lambda c: F.concat(F.substring(c, 1, 1), F.lit("***@"), F.substring_index(c, "@", -1)),
    "phone": lambda c: F.concat(F.lit("***-***-"), F.substring(c, -4, 4)),
    "date_of_birth": lambda c: F.concat(F.year(c).cast("string"), F.lit("-**-**")),
    "last_name": lambda c: F.concat(F.substring(c, 1, 1), F.lit(".")),
}
IDENTITY_COLUMNS = ["first_name", "last_name", "email", "phone", "date_of_birth"]

# ---------- who can see what (would be Entra ID groups in Fabric) ----------
ACCESS_POLICIES = {
    "admin@university.edu":       {"role": "admin",      "courses": "*",      "pii": True},
    "ayesha.khan@university.edu": {"role": "instructor", "courses": ["C001"], "pii": True},
    "analyst@university.edu":     {"role": "analyst",    "courses": "*",      "pii": False},
    "ml.service@university.edu":  {"role": "ml_service", "courses": "*",      "pii": False,
                                   "deny_columns": IDENTITY_COLUMNS},
}


def apply_masking(df):
    for col, mask in MASKS.items():
        if col in df.columns:
            df = df.withColumn(col, mask(F.col(col)))
    return df


def secure_read(spark, table: str, user: str):
    """Read a table the way `user` is allowed to see it: RLS -> CLS -> masking."""
    policy = ACCESS_POLICIES.get(user)
    if policy is None:
        raise PermissionError(f"{user} has no access policy")
    df = spark.table(table)

    if policy["courses"] != "*":  # ---- row-level security ----
        allowed = policy["courses"]
        if "course_id" in df.columns:
            df = df.filter(F.col("course_id").isin(allowed))
        elif "student_id" in df.columns:
            visible = spark.table("silver_enrollments").filter(F.col("course_id").isin(allowed))
            df = df.join(visible.select("student_id").distinct(), "student_id", "left_semi")

    deny = [c for c in policy.get("deny_columns", []) if c in df.columns]
    df = df.drop(*deny)  # ---- column-level security ----

    if not policy["pii"]:  # ---- dynamic data masking ----
        df = apply_masking(df)
    return df


def build_masked_student_dim(spark, cfg):
    df = spark.table("silver_students").select(
        F.sha2(F.concat(F.col("student_id"), F.lit("|edu-salt")), 256).alias("student_key"),
        "student_id", "first_name", "last_name", "email", "phone", "date_of_birth", "enrollment_date",
    )
    return write_table(apply_masking(df), "gold_dim_student", cfg)


# ---------- catalog / metadata tagging ----------
CATALOG = {
    "bronze_students":   {"layer": "bronze", "sensitivity": "Highly Confidential", "pii": "email,phone,date_of_birth,first_name,last_name"},
    "bronze_courses":    {"layer": "bronze", "sensitivity": "General", "pii": ""},
    "bronze_assignments": {"layer": "bronze", "sensitivity": "General", "pii": ""},
    "bronze_submissions": {"layer": "bronze", "sensitivity": "Confidential", "pii": ""},
    "silver_students":   {"layer": "silver", "sensitivity": "Highly Confidential", "pii": "email,phone,date_of_birth,first_name,last_name"},
    "silver_courses":    {"layer": "silver", "sensitivity": "General", "pii": ""},
    "silver_enrollments": {"layer": "silver", "sensitivity": "Confidential", "pii": ""},
    "silver_assignments": {"layer": "silver", "sensitivity": "General", "pii": ""},
    "silver_submissions": {"layer": "silver", "sensitivity": "Confidential", "pii": ""},
    "silver_student_assignment_status": {"layer": "silver", "sensitivity": "Confidential", "pii": ""},
    "silver_quarantine": {"layer": "silver", "sensitivity": "Highly Confidential", "pii": "raw_record"},
    "gold_course_submission_rate": {"layer": "gold", "sensitivity": "General", "pii": ""},
    "gold_student_overdue_metrics": {"layer": "gold", "sensitivity": "Confidential", "pii": "first_name,last_name"},
    "gold_assignment_summary": {"layer": "gold", "sensitivity": "General", "pii": ""},
    "gold_dim_student":  {"layer": "gold", "sensitivity": "Confidential (masked)", "pii": ""},
}


def apply_catalog(spark, cfg) -> list:
    """Tag every table with governance metadata and export a JSON data catalog."""
    entries = []
    for table, meta in CATALOG.items():
        props = {"layer": meta["layer"], "sensitivity": meta["sensitivity"],
                 "pii_columns": meta["pii"], "data_owner": "data-engineering@university.edu",
                 "domain": "education.assignments"}
        props_sql = ", ".join(f"'{k}' = '{v}'" for k, v in props.items())
        try:
            spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ({props_sql})")
        except Exception as exc:  # some local table formats don't support every property
            log.warning("could not tag %s: %s", table, exc)
        df = spark.table(table)
        entries.append({
            "table": table, **props,
            "row_count": df.count(),
            "columns": [{"name": f.name, "type": f.dataType.simpleString(),
                         "pii": f.name in meta["pii"].split(",")} for f in df.schema.fields],
        })
    out_dir = os.path.join(cfg.exports_fs, "catalog")
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "data_catalog.json"), "w", encoding="utf-8") as fh:
        json.dump(entries, fh, indent=2)
    return entries


def run_governance(spark, cfg) -> dict:
    build_masked_student_dim(spark, cfg)
    entries = apply_catalog(spark, cfg)
    return {"tables_tagged": len(entries)}

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 48, Finished, Available, Finished, False)

## 11. Orchestration: ingestion -> transformation -> validation -> export
One function runs the whole medallion flow, times each step, stops on the first
failure, and writes a run summary. The CLI (`scripts/run_pipeline.py`), the CI
pipelines and the notebook all call this same function.

In [47]:
import json
import os
import time
from datetime import datetime, timezone


ALL_STEPS = ["generate", "bronze", "silver", "gold", "quality", "governance", "export"]


def run_pipeline(spark, cfg, steps=None) -> dict:
    steps = steps or ALL_STEPS
    summary = {"started_at": datetime.now(timezone.utc).isoformat(), "as_of_date": cfg.as_of_date,
               "environment": "fabric" if cfg.is_fabric else "local", "steps": {}}

    actions = {
        "generate": lambda: write_source_files(cfg.landing_fs, cfg.seed),
        "bronze": lambda: run_bronze(spark, cfg),
        "silver": lambda: run_silver(spark, cfg),
        "gold": lambda: run_gold(spark, cfg),
        "quality": lambda: run_quality_checks(spark),
        "governance": lambda: run_governance(spark, cfg),
        "export": lambda: {
            "api": list(export_api_responses(spark, cfg)),
            "gold_csv": export_gold_csv(spark, cfg, list(GOLD_TABLES) + ["gold_dim_student"]),
        },
    }
    if "bronze" in steps:
        reset_local_lakehouse(cfg)

    for step in steps:
        t0 = time.time()
        log.info("=== step: %s ===", step)
        try:
            result = actions[step]()
            summary["steps"][step] = {"status": "success", "seconds": round(time.time() - t0, 2),
                                      "result": result}
        except Exception as exc:
            summary["steps"][step] = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}
            summary["status"] = "failed"
            _write_summary(cfg, summary)
            raise
    summary["status"] = "success"
    summary["finished_at"] = datetime.now(timezone.utc).isoformat()
    _write_summary(cfg, summary)
    return summary


def _write_summary(cfg, summary):
    os.makedirs(cfg.exports_fs, exist_ok=True)
    with open(os.path.join(cfg.exports_fs, "run_summary.json"), "w", encoding="utf-8") as fh:
        json.dump(summary, fh, indent=2, default=str)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 49, Finished, Available, Finished, False)

---
# Run the pipeline
## Start session

In [48]:
cfg = PipelineConfig()
spark = get_spark(cfg)
reset_local_lakehouse(cfg)
print('environment:', 'Fabric' if cfg.is_fabric else 'local', '| as of:', cfg.as_of_date)
print('landing  :', cfg.landing_spark)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 50, Finished, Available, Finished, False)

environment: Fabric | as of: 2026-10-15
landing  : Files/landing


## Step 1: Ingest simulated sources into `Files/landing`

In [49]:
for name, (path, rows) in write_source_files(cfg.landing_fs, cfg.seed).items():
    print(f'{name:12s} {rows:4d} rows -> {path}')

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 51, Finished, Available, Finished, False)

courses         6 rows -> /lakehouse/default/Files/landing/courses.csv
assignments    16 rows -> /lakehouse/default/Files/landing/assignments.csv
students       61 rows -> /lakehouse/default/Files/landing/students.json
submissions   314 rows -> /lakehouse/default/Files/landing/submissions.json


In [50]:
# Peek at the raw files exactly as received (note the dirty values)
show(read_raw(spark, f'{cfg.landing_spark}/students.json', 'json'), 5)
show(read_raw(spark, f'{cfg.landing_spark}/submissions.json', 'json'), 5)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 52, Finished, Available, Finished, False)

student_id,first_name,last_name,email,phone,date_of_birth,enrollment_date,enrolled_courses,updated_at
S0001,Liam,Chen,liam.chen1@university.edu,+1-555-132-1488,2001-01-18,2026-08-26,"[C001, C004]",2026-08-02T00:00:00Z
S0002,Ananya,Sharma,ananya.sharma2@university.edu,+1-555-325-8359,2006-08-10,2026-08-24,"[C001, C004]",2026-08-02T00:00:00Z
S0003,Aarav,Nair,aarav.nair3@university.edu,+1-555-320-6514,2001-02-22,2026-08-30,"[C001, C003, C005]",2026-08-02T00:00:00Z
S0004,Arjun,Chen,arjun.chen4@university.edu,+1-555-847-8527,2006-01-05,2026-08-29,"[C001, C003, C005]",2026-08-02T00:00:00Z
S0005,Arjun,Rossi,ARJUN.ROSSI5@UNIVERSITY.EDU,+1-555-296-2139,2000-07-06,2026-08-25,"[C001, C003, C004]",2026-08-02T00:00:00Z


submission_id,student_id,assignment_id,submitted_at,score,channel
SUB00293,S0057,Assignment_501,2026-10-06T17:59:00Z,60,mobile
SUB00259,S0014,Assignment_403,2026-10-13T21:00:00Z,94,web
SUB00115,S0025,Assignment_202,2026-10-04 06:59:00,72,mobile
SUB00083,S0010,Assignment_201,2026-09-20T08:59:00Z,71,lms_api
SUB00015,S0020,Assignment_101,2026-09-17T13:59:00Z,84,mobile


## Step 2: Bronze

In [51]:
run_bronze(spark, cfg)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 53, Finished, Available, Finished, False)

{'bronze_courses': 6,
 'bronze_assignments': 16,
 'bronze_students': 61,
 'bronze_submissions': 314}

In [52]:
show(spark.table('bronze_assignments'))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 54, Finished, Available, Finished, False)

assignment_id,course_id,title,due_date,max_score,_source_file,_ingested_at,_batch_id
Assignment_101,C001,Lab 1,2026-09-20 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_102,C001,Project Milestone 2,2026-10-04 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_103,C001,Final Project 3,2026-10-25 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_201,C002,Lab 1,2026-09-23 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_202,C002,Project Milestone 2,2026/10/07 23:59,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_203,C002,Final Project 3,2026-10-28 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_301,C003,Lab 1,2026-09-26 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_302,C003,Project Milestone 2,2026-10-10 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_303,C003,Final Project 3,2026-10-31 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10
Assignment_401,C004,Lab 1,2026-09-29 23:59:00,100,assignments.csv,2026-10-22 20:28:27.248970,2e273887dc10


## Step 3: Silver

In [53]:
run_silver(spark, cfg)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 55, Finished, Available, Finished, False)

{'silver_courses': 5,
 'silver_students': 60,
 'silver_enrollments': 152,
 'silver_assignments': 15,
 'silver_submissions': 279,
 'silver_quarantine': 5,
 'silver_student_assignment_status': 456}

Rejected records are **kept with a reason**, not silently dropped:

In [54]:
show(spark.table('silver_quarantine'))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 56, Finished, Available, Finished, False)

entity,record_key,reject_reason,raw_record,quarantined_at
assignments,Assignment_901,UNKNOWN_COURSE,"{""course_id"":""C999"",""assignment_id"":""Assignment_901"",""title"":""Legacy Quiz"",""due_at"":""2026-09-30T23:59:00.000Z"",""max_score"":50}",2026-10-22 20:29:32.917988
submissions,SUB00021,INVALID_TIMESTAMP,"{""student_id"":""S0030"",""course_id"":""C001"",""assignment_id"":""Assignment_101"",""submission_id"":""SUB00021"",""raw_score"":60.0,""channel"":""mobile"",""max_score"":100}",2026-10-22 20:29:32.917988
submissions,SUB00311,UNKNOWN_STUDENT,"{""student_id"":""S9999"",""course_id"":""C001"",""assignment_id"":""Assignment_102"",""submission_id"":""SUB00311"",""submitted_at"":""2026-10-01T10:00:00.000Z"",""raw_score"":88.0,""channel"":""lms_api"",""max_score"":100}",2026-10-22 20:29:32.917988
submissions,SUB00312,UNKNOWN_ASSIGNMENT,"{""student_id"":""S0002"",""assignment_id"":""Assignment_999"",""submission_id"":""SUB00312"",""submitted_at"":""2026-10-01T10:00:00.000Z"",""raw_score"":70.0,""channel"":""mobile""}",2026-10-22 20:29:32.917988
submissions,SUB00313,NOT_ENROLLED,"{""student_id"":""S0001"",""course_id"":""C005"",""assignment_id"":""Assignment_501"",""submission_id"":""SUB00313"",""submitted_at"":""2026-09-30T09:00:00.000Z"",""raw_score"":90.0,""channel"":""mobile"",""max_score"":100}",2026-10-22 20:29:32.917988


The student <-> assignment <-> submission mapping (one row per assignment a student owes):

In [55]:
show(spark.table('silver_student_assignment_status').orderBy('student_id', 'assignment_id'), 10)
show(spark.table('silver_student_assignment_status').groupBy('status').count().orderBy('status'))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 57, Finished, Available, Finished, False)

student_id,course_id,assignment_id,due_at,submission_id,first_submitted_at,last_submitted_at,attempt_count,score,is_due,is_submitted,is_late,status,days_overdue,as_of_date
S0001,C001,Assignment_101,2026-09-20 23:59:00,None,NaT,NaT,NaN,NaN,True,False,False,MISSING_OVERDUE,25,2026-10-15
S0001,C001,Assignment_102,2026-10-04 23:59:00,SUB00033,2026-10-01 23:59:00,2026-10-01 23:59:00,1.0,81.0,True,True,False,SUBMITTED_ON_TIME,0,2026-10-15
S0001,C001,Assignment_103,2026-10-25 23:59:00,SUB00068,2026-10-14 05:00:00,2026-10-14 05:00:00,1.0,69.0,False,True,False,SUBMITTED_ON_TIME,0,2026-10-15
S0001,C004,Assignment_401,2026-09-29 23:59:00,SUB00200,2026-09-28 02:59:00,2026-09-28 02:59:00,1.0,80.0,True,True,False,SUBMITTED_ON_TIME,0,2026-10-15
S0001,C004,Assignment_402,2026-10-13 23:59:00,SUB00233,2026-10-09 06:59:00,2026-10-09 06:59:00,1.0,85.0,True,True,False,SUBMITTED_ON_TIME,0,2026-10-15
S0001,C004,Assignment_403,2026-11-03 23:59:00,None,NaT,NaT,NaN,NaN,False,False,False,PENDING,0,2026-10-15
S0002,C001,Assignment_101,2026-09-20 23:59:00,None,NaT,NaT,NaN,NaN,True,False,False,MISSING_OVERDUE,25,2026-10-15
S0002,C001,Assignment_102,2026-10-04 23:59:00,SUB00034,2026-10-02 14:59:00,2026-10-02 14:59:00,1.0,74.0,True,True,False,SUBMITTED_ON_TIME,0,2026-10-15
S0002,C001,Assignment_103,2026-10-25 23:59:00,None,NaT,NaT,NaN,NaN,False,False,False,PENDING,0,2026-10-15
S0002,C004,Assignment_401,2026-09-29 23:59:00,SUB00202,2026-09-29 07:59:00,2026-09-29 17:59:00,2.0,79.0,True,True,False,SUBMITTED_ON_TIME,0,2026-10-15


status,count
MISSING_OVERDUE,57
PENDING,120
SUBMITTED_LATE,34
SUBMITTED_ON_TIME,245


## Step 4: Gold

In [56]:
run_gold(spark, cfg)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 58, Finished, Available, Finished, False)

{'gold_course_submission_rate': 5,
 'gold_student_overdue_metrics': 60,
 'gold_assignment_summary': 15}

---
# Part 2: Product questions
Each question is a `.sql` file. Below: the SQL, the result table, then the API envelope.

### Q1. Students who have not submitted `Assignment_101`

In [57]:
q1 = load_sql('queries/q1_students_not_submitted_assignment_101.sql',
              as_of_date=cfg.as_of_date, assignment_id='Assignment_101')
print(q1)
show(spark.sql(q1))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 59, Finished, Available, Finished, False)


-- Q1: Students who have NOT submitted "Assignment_101".
-- Only students enrolled in the course that owns the assignment are considered
-- (a student outside the course was never expected to submit it).
SELECT
    st.student_id,
    st.first_name,
    st.last_name,
    e.course_id,
    a.assignment_id,
    a.due_at,
    DATEDIFF(DATE'2026-10-15', a.due_at) AS days_past_due
FROM silver_enrollments e
JOIN silver_assignments a  ON a.course_id  = e.course_id
JOIN silver_students    st ON st.student_id = e.student_id
WHERE a.assignment_id = 'Assignment_101'
  AND NOT EXISTS (
        SELECT 1
        FROM silver_submissions sub
        WHERE sub.student_id    = e.student_id
          AND sub.assignment_id = a.assignment_id
  )
ORDER BY st.student_id



student_id,first_name,last_name,course_id,assignment_id,due_at,days_past_due
S0001,Liam,Chen,C001,Assignment_101,2026-09-20 23:59:00,25
S0002,Ananya,Sharma,C001,Assignment_101,2026-09-20 23:59:00,25
S0003,Aarav,Nair,C001,Assignment_101,2026-09-20 23:59:00,25
S0004,Arjun,Chen,C001,Assignment_101,2026-09-20 23:59:00,25
S0009,Ethan,Silva,C001,Assignment_101,2026-09-20 23:59:00,25
S0013,Mia,Sharma,C001,Assignment_101,2026-09-20 23:59:00,25
S0016,Ethan,Costa,C001,Assignment_101,2026-09-20 23:59:00,25
S0030,Yusuf,Rossi,C001,Assignment_101,2026-09-20 23:59:00,25
S0035,Fatima,Sharma,C001,Assignment_101,2026-09-20 23:59:00,25
S0046,Noah,Smith,C001,Assignment_101,2026-09-20 23:59:00,25


### Q2. Submission rate per course (%)

In [58]:
q2 = load_sql('queries/q2_submission_rate_per_course.sql')
print(q2)
show(spark.sql(q2))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 60, Finished, Available, Finished, False)


-- Q2: Submission rate per course (%), from the Gold table.
SELECT
    course_id,
    course_name,
    enrolled_students,
    assignments_due,
    expected_submissions,
    actual_submissions,
    submission_rate_pct,
    on_time_rate_pct
FROM gold_course_submission_rate
ORDER BY submission_rate_pct DESC, course_id



course_id,course_name,enrolled_students,assignments_due,expected_submissions,actual_submissions,submission_rate_pct,on_time_rate_pct
C004,Cloud Architecture on Azure,31,2,62,53,85.48,77.42
C003,Statistics for Analytics,29,2,58,49,84.48,74.14
C002,Machine Learning I,25,2,50,41,82.00,66.00
C001,Data Engineering Fundamentals,38,2,76,56,73.68,59.21
C005,Business Intelligence & Visualisation,29,1,29,19,65.52,51.72


### Q3. Overdue assignment count per student

In [59]:
q3 = load_sql('queries/q3_overdue_count_per_student.sql')
print(q3)
show(spark.sql(q3), 15)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 61, Finished, Available, Finished, False)


-- Q3: Overdue assignment count per student (past due, nothing submitted).
SELECT
    student_id,
    first_name,
    last_name,
    assignments_due,
    overdue_assignment_count,
    late_submission_count,
    max_days_overdue,
    risk_band
FROM gold_student_overdue_metrics
ORDER BY overdue_assignment_count DESC, late_submission_count DESC, student_id



student_id,first_name,last_name,assignments_due,overdue_assignment_count,late_submission_count,max_days_overdue,risk_band
S0055,Mia,Mehta,5,3,1,13,HIGH
S0003,Aarav,Nair,5,3,0,25,HIGH
S0004,Arjun,Chen,5,2,1,25,HIGH
S0041,Zara,Nair,6,2,1,19,HIGH
S0048,Fatima,Brown,5,2,1,13,HIGH
S0005,Arjun,Rossi,6,2,0,19,HIGH
S0010,Chloe,Silva,6,2,0,11,HIGH
S0029,Maya,Mehta,4,2,0,11,HIGH
S0035,Fatima,Sharma,6,2,0,25,HIGH
S0039,Mia,Haddad,5,2,0,16,HIGH


### API-style JSON output `{status, message, data}`

In [60]:
import json

for name in QUERIES:
    resp = run_query(spark, cfg, name)
    preview = {**resp, 'data': resp['data'][:2]}
    print(f'--- {name} ({len(resp["data"])} rows, first 2 shown) ---')
    print(json.dumps(preview, indent=2))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 62, Finished, Available, Finished, False)

--- students_not_submitted_assignment_101 (12 rows, first 2 shown) ---
{
  "status": "success",
  "message": "12 student(s) have not submitted Assignment_101",
  "data": [
    {
      "student_id": "S0001",
      "first_name": "Liam",
      "last_name": "Chen",
      "course_id": "C001",
      "assignment_id": "Assignment_101",
      "due_at": "2026-09-20T23:59:00",
      "days_past_due": 25
    },
    {
      "student_id": "S0002",
      "first_name": "Ananya",
      "last_name": "Sharma",
      "course_id": "C001",
      "assignment_id": "Assignment_101",
      "due_at": "2026-09-20T23:59:00",
      "days_past_due": 25
    }
  ]
}
--- submission_rate_per_course (5 rows, first 2 shown) ---
{
  "status": "success",
  "message": "Submission rate for 5 course(s) as of 2026-10-15",
  "data": [
    {
      "course_id": "C004",
      "course_name": "Cloud Architecture on Azure",
      "enrolled_students": 31,
      "assignments_due": 2,
      "expected_submissions": 62,
      "actual_submis

In [61]:
# The envelope holds on failure too
print(json.dumps(run_query(spark, cfg, 'unknown_question'), indent=2))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 63, Finished, Available, Finished, False)

{
  "status": "error",
  "message": "Unknown query 'unknown_question'. Available: ['overdue_count_per_student', 'students_not_submitted_assignment_101', 'submission_rate_per_course']",
  "data": []
}


---
# Part 3: Validation & export
## Data-quality gates

In [62]:
dq = run_quality_checks(spark)
show(spark.createDataFrame(dq), 50)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 64, Finished, Available, Finished, False)

check,failing_rows,passed
non_empty_bronze_students,0,True
non_empty_bronze_submissions,0,True
non_empty_silver_assignments,0,True
non_empty_silver_submissions,0,True
non_empty_silver_student_assignment_status,0,True
non_empty_gold_course_submission_rate,0,True
pk_silver_students,0,True
pk_silver_submissions,0,True
fk_enrollment_student,0,True
fk_submission_assignment,0,True


---
# Part 4 (bonus): Security & governance

In [63]:
run_governance(spark, cfg)
show(spark.table('gold_dim_student'), 5)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 65, Finished, Available, Finished, False)

student_key,student_id,first_name,last_name,email,phone,date_of_birth,enrollment_date
359e87ac6da0896ba97b7c8a3e220323e2c1f44569e4d0089a6a3eb5d4686a04,S0001,Liam,C.,l***@university.edu,***-***-1488,2001-**-**,2026-08-26
382a0160b0149e6df80ec3ef87190a3920e806a06c7e8e35ee3907c75c397084,S0002,Ananya,S.,a***@university.edu,***-***-8359,2006-**-**,2026-08-24
11ff2627de12c05f16ca97b7147dbc05f1807d3bc03a4d03ab1fc22da1c77f73,S0003,Aarav,N.,a***@university.edu,***-***-6514,2001-**-**,2026-08-30
50bf0145879b01c8508cec52bc48c2515be7843c80ea59f5dcf7f92ce2ff1ddd,S0004,Arjun,C.,a***@university.edu,***-***-8527,2006-**-**,2026-08-29
b115ccf0c37d30052e75bfb6cd347e6631b167fb5112a9d6c6cc72acce155a19,S0005,Arjun,R.,a***@university.edu,***-***-2139,2000-**-**,2026-08-25


Same table, three users: **RLS** (instructor sees only C001), **masking** (analyst), **CLS** (ML service gets no identity columns).

In [64]:
for user in ['ayesha.khan@university.edu', 'analyst@university.edu', 'ml.service@university.edu']:
    df = secure_read(spark, 'silver_students', user)
    print(f'\n{user}: {df.count()} rows, columns = {df.columns}')
    show(df, 3)

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 66, Finished, Available, Finished, False)


ayesha.khan@university.edu: 38 rows, columns = ['student_id', 'first_name', 'last_name', 'email', 'phone', 'date_of_birth', 'enrollment_date', 'updated_at']


student_id,first_name,last_name,email,phone,date_of_birth,enrollment_date,updated_at
S0001,Liam,Chen,liam.chen1@university.edu,+1-555-132-1488,2001-01-18,2026-08-26,2026-08-02
S0002,Ananya,Sharma,ananya.sharma2@university.edu,+1-555-325-8359,2006-08-10,2026-08-24,2026-08-02
S0003,Aarav,Nair,aarav.nair3@university.edu,+1-555-320-6514,2001-02-22,2026-08-30,2026-08-02



analyst@university.edu: 60 rows, columns = ['student_id', 'first_name', 'last_name', 'email', 'phone', 'date_of_birth', 'enrollment_date', 'updated_at']


student_id,first_name,last_name,email,phone,date_of_birth,enrollment_date,updated_at
S0001,Liam,C.,l***@university.edu,***-***-1488,2001-**-**,2026-08-26,2026-08-02
S0002,Ananya,S.,a***@university.edu,***-***-8359,2006-**-**,2026-08-24,2026-08-02
S0003,Aarav,N.,a***@university.edu,***-***-6514,2001-**-**,2026-08-30,2026-08-02



ml.service@university.edu: 60 rows, columns = ['student_id', 'enrollment_date', 'updated_at']


student_id,enrollment_date,updated_at
S0001,2026-08-26,2026-08-02
S0002,2026-08-24,2026-08-02
S0003,2026-08-30,2026-08-02


In [65]:
show(secure_read(spark, 'gold_course_submission_rate', 'ayesha.khan@university.edu'))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 67, Finished, Available, Finished, False)

course_id,course_name,department,instructor,enrolled_students,assignments_due,expected_submissions,actual_submissions,on_time_submissions,late_submissions,missing_submissions,avg_score,submission_rate_pct,on_time_rate_pct,_refreshed_at
C001,Data Engineering Fundamentals,Computer Science,Dr. Ayesha Khan,38,2,76,56,45,11,20,78.37,73.68,59.21,2026-10-22 20:29:53.203678


In [66]:
show(spark.sql('SHOW TBLPROPERTIES silver_students'))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 68, Finished, Available, Finished, False)

key,value
data_owner,data-engineering@university.edu
delta.feature.appendOnly,supported
delta.feature.invariants,supported
delta.minWriterVersion,7
delta.stats.extended.inject,true
domain,education.assignments
pii_columns,"email,phone,date_of_birth,first_name,last_name"
delta.enableDeletionVectors,true
delta.feature.deletionVectors,supported
delta.minReaderVersion,3


## Export Gold + API payloads to `Files/exports`

In [67]:
print(list(export_api_responses(spark, cfg)))
print(export_gold_csv(spark, cfg, list(GOLD_TABLES) + ['gold_dim_student']))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 69, Finished, Available, Finished, False)

['students_not_submitted_assignment_101', 'submission_rate_per_course', 'overdue_count_per_student']
['/lakehouse/default/Files/exports/gold/gold_course_submission_rate.csv', '/lakehouse/default/Files/exports/gold/gold_student_overdue_metrics.csv', '/lakehouse/default/Files/exports/gold/gold_assignment_summary.csv', '/lakehouse/default/Files/exports/gold/gold_dim_student.csv']


---
## One-call orchestration
Everything above in a single call, which is what the CLI, CI and a Fabric Data Pipeline / notebook schedule invoke:

In [68]:
summary = run_pipeline(spark, cfg)
print(json.dumps({k: v['status'] for k, v in summary['steps'].items()}, indent=2))

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 70, Finished, Available, Finished, False)

{
  "generate": "success",
  "bronze": "success",
  "silver": "success",
  "gold": "success",
  "quality": "success",
  "governance": "success",
  "export": "success"
}


In [69]:
import os, shutil

base = "/tmp/lakehouse_export"
shutil.rmtree(base, ignore_errors=True)
os.makedirs(f"{base}/tables", exist_ok=True)

# 1. Every Lakehouse table (bronze_*, silver_*, gold_*) -> one CSV each
for t in spark.catalog.listTables():
    if t.isTemporary:
        continue
    pdf = spark.table(t.name).toPandas()
    pdf.to_csv(f"{base}/tables/{t.name}.csv", index=False)
    print(f"{t.name:40s} {len(pdf):5d} rows")

# 2. Raw landing files + pipeline exports (API JSON, Gold CSVs, catalog, run summary)
for folder in ["landing", "exports"]:
    src = f"/lakehouse/default/Files/{folder}"
    if os.path.isdir(src):
        shutil.copytree(src, f"{base}/files/{folder}")

# 3. Zip it all and save the zip into the Lakehouse Files area
zip_path = shutil.make_archive("/tmp/lakehouse_export", "zip", base)
shutil.copy(zip_path, "/lakehouse/default/Files/lakehouse_export.zip")
print("Done -> Files/lakehouse_export.zip")

StatementMeta(, a10fe864-3225-4711-a56c-216ec626b63a, 71, Finished, Available, Finished, False)

bronze_courses                               6 rows
bronze_assignments                          16 rows
bronze_students                             61 rows
bronze_submissions                         314 rows
silver_courses                               5 rows
silver_enrollments                         152 rows
silver_students                             60 rows
silver_assignments                          15 rows
silver_submissions                         279 rows
silver_quarantine                            5 rows
silver_student_assignment_status           456 rows
gold_course_submission_rate                  5 rows
gold_student_overdue_metrics                60 rows
gold_assignment_summary                     15 rows
gold_dim_student                            60 rows
Done -> Files/lakehouse_export.zip
